# 🧠 EduTutor — Notebook 1: Dataset Generation Pipeline

**Project:** EduTutor-Unsloth &mdash; A Neurodiversity-Affirming AI Tutor  
**Competition:** [Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon)  
**Track:** Future of Education + Unsloth Tech Track  
**Model Target:** Gemma 4 E4B (4B params, 128K context, multimodal)  
**Inference:** Local model via Unsloth (no API key needed)  

---

## Purpose

This notebook generates a high-quality **synthetic conversational dataset** for fine-tuning Gemma 4 E4B into a neurodiversity-affirming tutor. Instead of training a generic "helpful assistant," we create data that teaches the model to:

1. **Facilitate productive struggle** — guide without giving answers away  
2. **Manage cognitive load** — use short sentences, chunked steps, and visual formatting  
3. **Co-regulate emotions** — detect frustration/shutdown and respond with empathy before returning to content  
4. **Adapt to neurodivergent profiles** — different strategies for ADHD, autism, dyslexia, and dyscalculia  

### Pedagogical Frameworks Encoded

| Framework | What It Does | How We Encode It |
|---|---|---|
| **UDL** (Universal Design for Learning) | Multiple means of engagement, representation, action | Tutor offers content in different formats |
| **ZPD** (Zone of Proximal Development) | "I Do, We Do, You Do" scaffolding | Tutor models, then collaborates, then lets student try |
| **CRA** (Concrete-Representational-Abstract) | Concrete → Visual → Symbolic progression | Math problems always start with tangible examples |
| **Orton-Gillingham** | Multisensory structured literacy | Reading tasks use phonemic awareness sequences |
| **Zones of Regulation** | Blue/Green/Yellow/Red emotional states | Tutor checks in on feelings and adapts pacing |
| **Spaced Repetition** | Review at increasing intervals | Tutor weaves past concepts into new lessons |

## 1. Environment Setup

We load the Gemma 4 model **locally** via Unsloth — no Google API key required. The same base model serves as our "teacher" for generating synthetic data.

> **⚠️ Self-Distillation Trade-off:** Using the same E4B model as both teacher (data generator) and student (model being fine-tuned) creates a self-distillation loop, which can reduce data diversity. For higher-quality data, upgrade the teacher to `google/gemma-4-12b-it` (24GB+ VRAM) or `google/gemma-4-27b-it` (40GB+ VRAM). The code below works with any model loaded via `load_teacher_model()`.

In [1]:
%%capture
!pip install -U unsloth
!pip install -q datasets jsonlines tqdm

In [4]:
# --- Castalia/EduTutor Colab + Drive Setup ---
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path.cwd()

# Persist HuggingFace/Unsloth downloads and EduTutor outputs on Drive in Colab.
CACHE_DIR = DRIVE_ROOT / "models"
EDUTUTOR_ROOT = DRIVE_ROOT / "castalia-hackathons" / "edututor"
DATA_DIR = EDUTUTOR_ROOT / "data"
MODEL_DIR = EDUTUTOR_ROOT / "models"
EVAL_DIR = EDUTUTOR_ROOT / "eval"
SESSION_DIR = EDUTUTOR_ROOT / "sessions"

for path in [CACHE_DIR, DATA_DIR, MODEL_DIR, EVAL_DIR, SESSION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["EDUTUTOR_CACHE_DIR"] = str(CACHE_DIR)

repo_candidates = [
    Path.cwd(),
    Path("/content/castalia"),
    DRIVE_ROOT / "castalia",
]
module_candidates = []
for candidate in repo_candidates:
    module_candidates.extend([
        candidate,
        candidate / "hackathons" / "gemma-4-good-hackathon" / "EduTutor",
    ])

module_dir = next((path for path in module_candidates if (path / "local_model.py").exists()), None)

if module_dir is None and IN_COLAB:
    repo_dir = Path("/content/castalia")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/cyruslayo/castalia.git", str(repo_dir)],
            check=True,
        )
    module_dir = repo_dir / "hackathons" / "gemma-4-good-hackathon" / "EduTutor"

if module_dir is None or not (module_dir / "local_model.py").exists():
    raise FileNotFoundError(
        "Could not find local_model.py. Run from the EduTutor folder, from the castalia repo root, "
        "or use the Colab bootstrap to clone castalia."
    )

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

print(f"EduTutor module path: {module_dir}")
print(f"Model cache: {CACHE_DIR}")
print(f"EduTutor outputs: {EDUTUTOR_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
EduTutor module path: /content/castalia/hackathons/gemma-4-good-hackathon/EduTutor
Model cache: /content/drive/MyDrive/models
EduTutor outputs: /content/drive/MyDrive/castalia-hackathons/edututor


In [5]:
import json
import random
import re
import os
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Literal
from enum import Enum

import jsonlines
from tqdm.auto import tqdm

# ---------- Local Model Setup ----------
from local_model import load_teacher_model, generate_text, EDUTUTOR_SYSTEM_PROMPT

# Load Gemma 4 E4B locally via Unsloth (4-bit quantization)
# The previously referenced 12b and 27b models do not exist on the Hub.
model, tokenizer = load_teacher_model("google/gemma-4-31B-it", cache_dir=CACHE_DIR)

OUTPUT_DIR = DATA_DIR
OUTPUT_DIR.mkdir(exist_ok=True)

random.seed(42)
print(f"✅ Ready. Teacher model loaded locally.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
🖥️  GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (102.0 GB)
🗄️  Cache: /content/drive/MyDrive/models
📦 Loading google/gemma-4-31B-it (4-bit=True)...
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

✅ Model loaded: google/gemma-4-31B-it
   Parameters: 31,273,086,512
✅ Ready. Teacher model loaded locally.


## 2. Neurodivergent Learner Profiles

We define four learner archetypes grounded in clinical and educational research. Each profile specifies:
- **Cognitive strengths** — what the student does well (strengths-based, not deficit-based)
- **Learning barriers** — specific challenges the tutor must accommodate
- **Preferred strategies** — evidence-based interventions that work for this profile
- **Emotional patterns** — common frustration triggers and regulation needs

In [6]:
@dataclass
class LearnerProfile:
    """A neurodivergent learner archetype for synthetic data generation."""
    name: str
    condition: str
    age_range: str
    strengths: list[str]
    barriers: list[str]
    preferred_strategies: list[str]
    emotional_triggers: list[str]
    example_behaviors: list[str]


PROFILES = [
    LearnerProfile(
        name="ADHD Learner",
        condition="ADHD (combined type)",
        age_range="8-14",
        strengths=[
            "Creative and divergent thinking",
            "High energy and enthusiasm when engaged",
            "Strong verbal skills and storytelling",
            "Ability to hyperfocus on interesting topics",
            "Quick pattern recognition"
        ],
        barriers=[
            "Weak working memory — forgets multi-step instructions",
            "Executive dysfunction — struggles to start, plan, and sequence tasks",
            "Time blindness — cannot estimate how long tasks take",
            "Sustained attention is inconsistent — focus drifts after 5-10 minutes",
            "Emotional dysregulation — frustration escalates rapidly"
        ],
        preferred_strategies=[
            "Break tasks into 2-3 minute micro-chunks",
            "Use novelty and interleaving to maintain engagement",
            "Provide external working memory aids (checklists, step trackers)",
            "Celebrate effort immediately — delayed rewards don't work",
            "Use movement and physical analogies",
            "Spaced repetition with short flashcard bursts"
        ],
        emotional_triggers=[
            "Being told to 'just focus' or 'try harder'",
            "Multi-step problems presented all at once",
            "Feeling behind compared to peers",
            "Repetitive drill without novelty"
        ],
        example_behaviors=[
            "'This is boring, can we do something else?'",
            "'Wait, what was step one again? I forgot.'",
            "'I already know this!' (skipping ahead without mastery)",
            "Gives correct answer by guessing but can't explain the process",
            "Gets frustrated and says 'I'm stupid' after one mistake"
        ]
    ),
    LearnerProfile(
        name="Autistic Learner",
        condition="Autism Spectrum (Level 1)",
        age_range="8-14",
        strengths=[
            "Excellent long-term memory for facts and systems",
            "Deep focus on special interests",
            "Strong logical and rule-based thinking",
            "Visual-spatial processing often strong",
            "Honest and direct communication style"
        ],
        barriers=[
            "Difficulty with ambiguous or open-ended instructions",
            "Literal interpretation — misses implied meaning",
            "Transitions between topics cause anxiety",
            "Sensory overload from too much visual/textual information at once",
            "Struggles with inferential comprehension in reading"
        ],
        preferred_strategies=[
            "Provide explicit, unambiguous instructions with clear structure",
            "Use visual schedules and 'First-Then' sequencing",
            "Connect new topics to the student's special interests",
            "Warn before transitions: 'We're going to switch topics in a moment'",
            "Use concrete examples before abstract rules",
            "Social narratives for any collaborative scenarios"
        ],
        emotional_triggers=[
            "Unexpected changes to routine or topic",
            "Ambiguous questions like 'What do you think?'",
            "Being wrong feels catastrophic — perfectionism",
            "Sensory overwhelm from dense text blocks"
        ],
        example_behaviors=[
            "'That doesn't make sense. You said X but now you're saying Y.'",
            "'Can you be more specific about what you want me to do?'",
            "Answers the question literally, missing the intended question",
            "Shuts down completely after making an error",
            "Enthusiastically connects lesson to a special interest"
        ]
    ),
    LearnerProfile(
        name="Dyslexic Learner",
        condition="Dyslexia",
        age_range="8-14",
        strengths=[
            "Strong spatial reasoning and 3D thinking",
            "Creative problem-solving and big-picture thinking",
            "Excellent verbal reasoning and oral comprehension",
            "Strong narrative and storytelling ability",
            "Good at reasoning by analogy"
        ],
        barriers=[
            "Decoding written text is slow and effortful",
            "Phonemic awareness gaps — confuses similar sounds",
            "Spelling is inconsistent and frustrating",
            "Reading fluency is significantly below oral ability",
            "Dense text walls cause cognitive fatigue quickly"
        ],
        preferred_strategies=[
            "Use Orton-Gillingham structured literacy: explicit phonics progression",
            "Keep text short — max 2 sentences at a time",
            "Use bullet points and clear visual formatting",
            "Connect sounds to visual and kinesthetic cues",
            "Never ask the student to read aloud without warning",
            "Allow oral responses as an alternative to written ones"
        ],
        emotional_triggers=[
            "Being asked to read long passages",
            "Spelling corrections presented harshly",
            "Comparing reading speed to peers",
            "Dense instructions without formatting"
        ],
        example_behaviors=[
            "'Can you just tell me what it says?'",
            "Confuses b/d, p/q in written responses",
            "Gives brilliant verbal answers but can't write them down",
            "Avoids reading tasks entirely",
            "Says 'I hate reading' but loves audiobooks"
        ]
    ),
    LearnerProfile(
        name="Dyscalculic Learner",
        condition="Dyscalculia",
        age_range="8-14",
        strengths=[
            "Strong reading and verbal comprehension",
            "Good at qualitative reasoning and comparison",
            "Creative writing and artistic expression",
            "Understands concepts when presented visually",
            "Strong social and emotional intelligence"
        ],
        barriers=[
            "Number sense is weak — struggles with magnitude and estimation",
            "Procedural memory for math algorithms is unreliable",
            "Math anxiety severely reduces working memory capacity",
            "Cannot retain multiplication tables despite drilling",
            "Confuses operation signs and place values"
        ],
        preferred_strategies=[
            "Use CRA: start with concrete objects, then pictures, then numbers",
            "Always connect math to real-world, tangible scenarios",
            "Provide reference sheets — don't require memorization",
            "Use consistent language across all representations",
            "Reduce cognitive load: one operation at a time",
            "Color-code place values and operation signs"
        ],
        emotional_triggers=[
            "Timed math tests or speed drills",
            "Being told 'this is easy' when it isn't for them",
            "Abstract equations without context",
            "Seeing peers finish quickly"
        ],
        example_behaviors=[
            "'I just can't do math. My brain doesn't work that way.'",
            "Counts on fingers even for basic addition",
            "Gets the right answer with objects but wrong with numbers",
            "Freezes when seeing a word problem",
            "Says 'I don't know' before even trying — learned helplessness"
        ]
    )
]

print(f"Defined {len(PROFILES)} learner profiles:")
for p in PROFILES:
    print(f"  • {p.name} ({p.condition}) — {len(p.preferred_strategies)} strategies")

Defined 4 learner profiles:
  • ADHD Learner (ADHD (combined type)) — 6 strategies
  • Autistic Learner (Autism Spectrum (Level 1)) — 6 strategies
  • Dyslexic Learner (Dyslexia) — 6 strategies
  • Dyscalculic Learner (Dyscalculia) — 6 strategies


## 3. Academic Scenarios

We define specific academic scenarios across subjects and grade levels. Each scenario includes the subject, topic, grade level, and a realistic problem the student is working on.

In [7]:
@dataclass
class AcademicScenario:
    """A specific academic task the student is working on."""
    subject: str
    topic: str
    grade_level: str
    problem: str
    common_misconceptions: list[str]


SCENARIOS = [
    # --- MATH ---
    AcademicScenario(
        subject="Math", topic="Fractions", grade_level="Grade 4",
        problem="Add 1/4 + 2/4 and explain what the answer means.",
        common_misconceptions=["Adding numerators AND denominators (1/4 + 2/4 = 3/8)", "Not understanding what a fraction represents"]
    ),
    AcademicScenario(
        subject="Math", topic="Multiplication", grade_level="Grade 3",
        problem="What is 6 × 7? Can you show how you figured it out?",
        common_misconceptions=["Confusing multiplication with addition", "memorizing without understanding grouping"]
    ),
    AcademicScenario(
        subject="Math", topic="Word Problems", grade_level="Grade 5",
        problem="A store has 48 apples. They put them equally into 6 bags. How many apples in each bag?",
        common_misconceptions=["Not knowing which operation to use", "Being overwhelmed by the text before extracting numbers"]
    ),
    AcademicScenario(
        subject="Math", topic="Place Value", grade_level="Grade 3",
        problem="What is the value of the 5 in the number 352?",
        common_misconceptions=["Saying '5' instead of '50'", "Confusing the digit with its place value"]
    ),
    # --- READING ---
    AcademicScenario(
        subject="Reading", topic="Phonics — Digraphs", grade_level="Grade 2",
        problem="What sound do the letters 'sh' make together? Can you think of three words that start with 'sh'?",
        common_misconceptions=["Pronouncing each letter separately", "confusing sh/ch/th"]
    ),
    AcademicScenario(
        subject="Reading", topic="Reading Comprehension", grade_level="Grade 4",
        problem="Read this short passage: 'Sam looked at the dark clouds. He grabbed his umbrella and ran inside.' Why did Sam grab the umbrella?",
        common_misconceptions=["Literal answer without inference", "'Because it was there' instead of inferring rain"]
    ),
    AcademicScenario(
        subject="Reading", topic="Vocabulary — Context Clues", grade_level="Grade 5",
        problem="In the sentence 'The enormous elephant blocked the entire path', what does 'enormous' mean? How do you know?",
        common_misconceptions=["Guessing without using context", "Saying 'I don't know that word' and shutting down"]
    ),
    # --- WRITING ---
    AcademicScenario(
        subject="Writing", topic="Sentence Structure", grade_level="Grade 3",
        problem="Write a sentence about your favorite animal. Make sure it has a capital letter at the start and a period at the end.",
        common_misconceptions=["Run-on sentences", "Forgetting capitals/periods", "Frustration with spelling blocking idea expression"]
    ),
    AcademicScenario(
        subject="Writing", topic="Paragraph Organization", grade_level="Grade 5",
        problem="Write a short paragraph about why recess is important. Include a topic sentence, two supporting details, and a closing sentence.",
        common_misconceptions=["No topic sentence — jumping into details", "Ideas present but no logical order"]
    ),
    # --- SCIENCE ---
    AcademicScenario(
        subject="Science", topic="Water Cycle", grade_level="Grade 4",
        problem="Can you explain what happens to water in a puddle on a sunny day? Where does it go?",
        common_misconceptions=["'It disappears'", "Not connecting evaporation to the sun's heat"]
    ),
    AcademicScenario(
        subject="Science", topic="Plant Growth", grade_level="Grade 3",
        problem="What three things does a plant need to grow?",
        common_misconceptions=["Forgetting sunlight or water", "Saying 'dirt' instead of understanding nutrients/soil"]
    ),
    AcademicScenario(
        subject="Science", topic="States of Matter", grade_level="Grade 5",
        problem="Is steam a solid, liquid, or gas? How do you know?",
        common_misconceptions=["Confusing steam with smoke", "Thinking gases are always invisible"]
    ),
]

print(f"Defined {len(SCENARIOS)} academic scenarios across subjects:")
for subj in set(s.subject for s in SCENARIOS):
    count = sum(1 for s in SCENARIOS if s.subject == subj)
    print(f"  • {subj}: {count} scenarios")

Defined 12 academic scenarios across subjects:
  • Math: 4 scenarios
  • Reading: 3 scenarios
  • Science: 3 scenarios
  • Writing: 2 scenarios


## 4. Emotional State Scenarios

Based on the **Zones of Regulation** framework, we define the emotional states a student might be in during a session. The tutor must detect these and respond appropriately — content delivery pauses when the student is dysregulated.

In [8]:
class Zone(str, Enum):
    """Zones of Regulation — student emotional/arousal state."""
    GREEN = "green"    # Regulated, ready to learn
    YELLOW = "yellow"  # Heightened — frustrated, anxious, wiggly
    RED = "red"        # Overwhelmed — rage, panic, meltdown
    BLUE = "blue"      # Low arousal — sad, tired, bored, shutdown


ZONE_STUDENT_CUES = {
    Zone.GREEN: [
        "Okay, I think I understand. Let me try.",
        "That makes sense! What's next?",
        "I got it right! Can we do another one?",
        "Hmm, let me think about this...",
    ],
    Zone.YELLOW: [
        "Ugh, this is hard. I keep getting it wrong.",
        "Can we just skip this one?",
        "I don't get it and I've been trying!",
        "This is taking forever.",
        "Why do I have to learn this anyway?",
        "Wait, you're going too fast!",
    ],
    Zone.RED: [
        "I HATE this! I'm done!",
        "I can't do ANYTHING right! I'm so stupid!",
        "LEAVE ME ALONE!",
        "I want to throw this computer out the window!",
        "Everything is terrible and I want to quit!",
    ],
    Zone.BLUE: [
        "I don't care anymore.",
        "Whatever.",
        "I don't know.",
        "...",
        "Can I just stop?",
        "I'm tired.",
    ]
}

ZONE_TUTOR_STRATEGIES = {
    Zone.GREEN: "Continue teaching. The student is regulated and ready to learn. Use scaffolding and productive struggle.",
    Zone.YELLOW: "Pause the academic content. Acknowledge the frustration ('I can hear this is really hard for you right now, and that's okay.'). Offer a choice: take a brain break, switch to an easier warm-up, or try a different approach. Then return to the task with reduced difficulty.",
    Zone.RED: "STOP all academic demands immediately. This is a crisis state. Your only job is co-regulation: validate their feelings ('It makes sense that you feel that way — this really IS hard'), offer a calming strategy (deep breaths, counting, or just sitting quietly), and wait. Do NOT try to teach. Do NOT say 'calm down.' Only return to learning when the student signals readiness.",
    Zone.BLUE: "Gently re-engage. The student has withdrawn. Acknowledge their state without pressure ('It seems like your energy is low right now — that's okay'). Offer the easiest possible win to rebuild momentum. Connect to an interest they care about. Keep expectations minimal until they re-engage."
}

print("Zones of Regulation student cues defined:")
for zone, cues in ZONE_STUDENT_CUES.items():
    print(f"  {zone.value.upper()}: {len(cues)} example student utterances")

Zones of Regulation student cues defined:
  GREEN: 4 example student utterances
  YELLOW: 6 example student utterances
  RED: 5 example student utterances
  BLUE: 6 example student utterances


## 5. System Prompt — The EduTutor Persona

Imported from `local_model.py` as the **single source of truth**. All 4 notebooks use the same constant.

In [9]:
# Use the shared system prompt from local_model.py (single source of truth)
SYSTEM_PROMPT = EDUTUTOR_SYSTEM_PROMPT

print(f"System prompt loaded: {len(SYSTEM_PROMPT)} characters")

System prompt loaded: 2615 characters


## 6. SFT Data Generation: Productive Struggle Conversations

We use the locally loaded model as a teacher to simulate realistic tutoring conversations. For each combination of (learner profile × academic scenario × emotional zone), we generate a multi-turn dialogue showing ideal tutor behavior.

### Generation Strategy
- **Teacher Model** (local Gemma 4) generates a full conversation transcript
- We parse it into alternating `user` / `assistant` turns
- The `system` prompt is prepended to every example

In [10]:
def build_sft_generation_prompt(
    profile: LearnerProfile,
    scenario: AcademicScenario,
    zone: Zone,
) -> str:
    """Build a prompt that asks the teacher model to generate a realistic tutoring conversation."""
    # Pre-build the strategies list outside the f-string for readability
    strategies_text = '\n'.join('- ' + s for s in profile.preferred_strategies[:4])

    return f"""You are an expert educational content designer specializing in neurodiversity-affirming pedagogy. Your task is to generate a REALISTIC multi-turn conversation (6-10 turns) between:

**Student:** A {profile.age_range} year old with {profile.condition}.
- Strengths: {', '.join(profile.strengths[:3])}
- Barriers: {', '.join(profile.barriers[:3])}
- Current emotional state: {zone.value.upper()} zone ({ZONE_TUTOR_STRATEGIES[zone][:100]}...)
- Typical behavior: {random.choice(profile.example_behaviors)}

**Tutor (EduTutor):** A warm, patient AI tutor. Must follow these strategies:
{strategies_text}

**Academic Task:**
- Subject: {scenario.subject} — {scenario.topic} (Grade level: {scenario.grade_level})
- Problem: {scenario.problem}
- Common misconceptions to watch for: {', '.join(scenario.common_misconceptions)}

**Requirements:**
1. The student starts in the {zone.value.upper()} zone. If not GREEN, the tutor must address the emotional state BEFORE attempting any academic work.
2. The tutor must NEVER give the answer directly. Always scaffold toward discovery.
3. The tutor keeps sentences SHORT (under 15 words when possible) and uses bullet points.
4. The tutor checks in on the student's feelings at least once during the conversation.
5. If the student makes a mistake, the tutor validates the attempt before correcting.
6. The conversation should feel natural and warm, not robotic.

**Output Format:**
Format the conversation EXACTLY like this, with no other text:

STUDENT: [student's message]
TUTOR: [tutor's response]
STUDENT: [student's reply]
TUTOR: [tutor's response]
...

Generate the conversation now."""


# Quick test
test_prompt = build_sft_generation_prompt(PROFILES[0], SCENARIOS[0], Zone.YELLOW)
print(test_prompt[:500] + "\n...")

You are an expert educational content designer specializing in neurodiversity-affirming pedagogy. Your task is to generate a REALISTIC multi-turn conversation (6-10 turns) between:

**Student:** A 8-14 year old with ADHD (combined type).
- Strengths: Creative and divergent thinking, High energy and enthusiasm when engaged, Strong verbal skills and storytelling
- Barriers: Weak working memory — forgets multi-step instructions, Executive dysfunction — struggles to start, plan, and sequence tasks, 
...


In [11]:
def parse_conversation(raw_text: str) -> list[dict] | None:
    """Parse a STUDENT/TUTOR conversation into SFT-compatible message list."""
    messages = []
    # Split on STUDENT: or TUTOR: markers
    pattern = r'(STUDENT|TUTOR):\s*'
    parts = re.split(pattern, raw_text.strip())

    # parts will be: ['', 'STUDENT', 'msg1', 'TUTOR', 'msg2', ...]
    i = 1  # skip empty first element
    while i < len(parts) - 1:
        role_label = parts[i].strip()
        content = parts[i + 1].strip()

        if role_label == "STUDENT":
            messages.append({"role": "user", "content": content})
        elif role_label == "TUTOR":
            messages.append({"role": "assistant", "content": content})
        i += 2

    # Validate: must start with user, alternate, have at least 4 turns
    if len(messages) < 4:
        return None
    if messages[0]["role"] != "user":
        return None

    return messages


def format_sft_example(messages: list[dict]) -> dict:
    """Format a parsed conversation into the Gemma chat template structure."""
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            *messages
        ]
    }


# Verify parser works
test_conv = """STUDENT: I don't get fractions. They're dumb.
TUTOR: I hear you — fractions can feel tricky! 💪 But you know what? You already use fractions every day. Have you ever shared a pizza?
STUDENT: Yeah, I guess.
TUTOR: Perfect! If you cut a pizza into 4 slices and eat 1, you ate 1 out of 4 slices. We write that as 1/4. The bottom number (4) tells us how many total slices. The top number (1) tells us how many you ate. Does that make sense so far?
STUDENT: Oh, so the bottom is the total? I think so.
TUTOR: Exactly right! ⭐ The bottom = total pieces, the top = pieces you're talking about. Now, if you ate 1 slice and your friend ate 2 slices of that same pizza, how many slices were eaten altogether?"""

parsed = parse_conversation(test_conv)
print(f"Parsed {len(parsed)} turns")
for m in parsed:
    print(f"  [{m['role']}] {m['content'][:60]}...")

Parsed 6 turns
  [user] I don't get fractions. They're dumb....
  [assistant] I hear you — fractions can feel tricky! 💪 But you know what?...
  [user] Yeah, I guess....
  [assistant] Perfect! If you cut a pizza into 4 slices and eat 1, you ate...
  [user] Oh, so the bottom is the total? I think so....
  [assistant] Exactly right! ⭐ The bottom = total pieces, the top = pieces...


In [12]:
import traceback

def generate_sft_example(
    profile: LearnerProfile,
    scenario: AcademicScenario,
    zone: Zone,
) -> dict | None:
    """Generate a single SFT training example using the LOCAL teacher model."""
    prompt_text = build_sft_generation_prompt(profile, scenario, zone)
    # Pass the multimodal content list directly to generate_text
    multimodal_prompt = [{"type": "text", "text": prompt_text}]

    try:
        raw = generate_text(
            multimodal_prompt,
            max_new_tokens=2048,
            temperature=0.7,  # Lower temp for SFT = more coherent conversations
            top_p=0.95,
        )

        messages = parse_conversation(raw)

        if messages is None:
            return None

        example = format_sft_example(messages)
        example["metadata"] = {
            "profile": profile.condition,
            "subject": scenario.subject,
            "topic": scenario.topic,
            "zone": zone.value,
            "num_turns": len(messages),
        }
        return example

    except Exception as e:
        print(f"  [ERROR] {e}")
        traceback.print_exc()
        return None


print("SFT generation function defined (local inference with traceback).")

SFT generation function defined (local inference with traceback).


### 6.1 Generate the Full SFT Dataset

We generate conversations for every combination of:
- 4 learner profiles × 12 scenarios × 4 emotional zones = **192 base combinations**
- With 3 variations each = **~576 conversations**

This gives us a rich, diverse dataset covering all profiles, subjects, and emotional states.

> **Note:** Since we're running locally on GPU (not calling an API), generation is sequential.
> On a T4 GPU with the E4B model, expect ~30 seconds per conversation, so ~5 hours total.
> Reduce `VARIATIONS_PER_COMBO` to 1 for a faster first pass (~1.5 hours).

In [13]:
# ---------- Configuration ----------
VARIATIONS_PER_COMBO = 1     # generations per (profile, scenario, zone)
SFT_OUTPUT_FILE = OUTPUT_DIR / "sft_dataset.jsonl"


def generate_sft_dataset():
    """Generate the full SFT dataset across all combinations (sequential, local GPU)."""
    combos = []
    for profile in PROFILES:
        for scenario in SCENARIOS:
            for zone in Zone:
                for _ in range(VARIATIONS_PER_COMBO):
                    combos.append((profile, scenario, zone))

    random.shuffle(combos)
    print(f"Total generation tasks: {len(combos)}")

    results = []
    failed = 0

    for profile, scenario, zone in tqdm(combos, desc="Generating SFT examples"):
        result = generate_sft_example(profile, scenario, zone)
        if result is not None:
            results.append(result)
        else:
            failed += 1

        # Save checkpoint every 50 examples
        if len(results) % 50 == 0 and len(results) > 0:
            with jsonlines.open(SFT_OUTPUT_FILE, mode='w') as writer:
                writer.write_all(results)
            print(f"  💾 Checkpoint: {len(results)} saved, {failed} failed")

    # Final save
    with jsonlines.open(SFT_OUTPUT_FILE, mode='w') as writer:
        writer.write_all(results)

    print(f"\n✅ Generated {len(results)} SFT examples ({failed} failed)")
    print(f"   Saved to: {SFT_OUTPUT_FILE}")

    # Stats
    by_profile = {}
    by_zone = {}
    by_subject = {}
    for r in results:
        m = r["metadata"]
        by_profile[m["profile"]] = by_profile.get(m["profile"], 0) + 1
        by_zone[m["zone"]] = by_zone.get(m["zone"], 0) + 1
        by_subject[m["subject"]] = by_subject.get(m["subject"], 0) + 1

    print(f"\nBy Profile: {json.dumps(by_profile, indent=2)}")
    print(f"By Zone: {json.dumps(by_zone, indent=2)}")
    print(f"By Subject: {json.dumps(by_subject, indent=2)}")

    return results


sft_data = generate_sft_dataset()


Total generation tasks: 192


Generating SFT examples:   0%|          | 0/192 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


KeyboardInterrupt: 

### 6.2 Data Quality Scoring & Filtering

Before proceeding with DPO generation, we apply a **multi-dimensional quality scoring rubric** to every SFT example.
Each conversation is scored on 5 dimensions (1–5 scale) grounded in our pedagogical framework:

| Dimension | What It Measures | Why It Matters |
|---|---|---|
| Pedagogical Accuracy | Tutor avoids giving answers directly | Core productive-struggle principle |
| Cognitive Load | Short, chunked sentences | UDL + working-memory accommodations |
| Emotional Validation | Empathy phrases present | Zones of Regulation co-regulation |
| Formatting Adherence | Bullet points / structure used | Visual accessibility for dyslexia |
| Turn Alternation | Strict user↔assistant alternation | Valid chat-template structure |

Examples with an **overall score < 3.5** are filtered out to ensure training data quality.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns


def score_sft_quality(example: dict) -> dict:
    """Score conversation quality on 5 dimensions (1-5 scale)."""
    messages = example.get("messages", [])
    tutor_msgs = [m["content"] for m in messages if m["role"] == "assistant"]
    all_text = " ".join(tutor_msgs).lower()

    # 1. Pedagogical accuracy — penalise answer-giving phrases
    answer_phrases = ["the answer is", "so the answer", "it's definitely"]
    hits = sum(1 for p in answer_phrases if p in all_text)
    pedagogical = max(1, 5 - hits)

    # 2. Cognitive load — average sentence length across tutor responses
    import re as _re
    sentences = _re.split(r'[.!?]+', all_text)
    word_counts = [len(s.split()) for s in sentences if len(s.split()) > 2]
    avg_sent = np.mean(word_counts) if word_counts else 20
    if avg_sent < 15:
        cognitive = 5
    elif avg_sent < 20:
        cognitive = 4
    else:
        cognitive = 1

    # 3. Emotional validation — empathy / validation phrases
    validation_phrases = ["i can hear", "i understand", "that's okay", "valid", "feel"]
    emotional = 5 if any(p in all_text for p in validation_phrases) else 1

    # 4. Formatting adherence — proportion of tutor turns with structure
    structure_markers = ["\u2022", "\u25aa", "- ", "1.", "2."]
    structured = sum(1 for t in tutor_msgs if any(m in t for m in structure_markers))
    ratio = structured / len(tutor_msgs) if tutor_msgs else 0
    formatting = 5 if ratio > 0.3 else 3

    # 5. Turn alternation — consecutive messages must differ in role
    non_sys = [m for m in messages if m["role"] != "system"]
    alternates = all(
        non_sys[i]["role"] != non_sys[i + 1]["role"]
        for i in range(len(non_sys) - 1)
    ) if len(non_sys) > 1 else False
    turn_alt = 5 if alternates else 1

    scores = {
        "pedagogical_accuracy": pedagogical,
        "cognitive_load": cognitive,
        "emotional_validation": emotional,
        "formatting_adherence": formatting,
        "turn_alternation": turn_alt,
    }
    scores["overall"] = np.mean(list(scores.values()))
    return scores


# ── Score every SFT example ─────────────────────────────────────────
QUALITY_THRESHOLD = 3.5

scored = []
for ex in sft_data:
    ex["quality_scores"] = score_sft_quality(ex)
    scored.append(ex)

before_count = len(scored)
sft_data_filtered = [ex for ex in scored if ex["quality_scores"]["overall"] >= QUALITY_THRESHOLD]
after_count = len(sft_data_filtered)
removed = before_count - after_count

print(f"\n{'=' * 60}")
print(f"📊 QUALITY SCORING RESULTS")
print(f"{'=' * 60}")
print(f"  Total examples scored : {before_count}")
print(f"  Passed (≥ {QUALITY_THRESHOLD})        : {after_count}")
print(f"  Filtered out          : {removed}  ({removed/before_count*100:.1f}%)" if before_count else "")

# Per-dimension stats
dims = ["pedagogical_accuracy", "cognitive_load", "emotional_validation",
        "formatting_adherence", "turn_alternation"]
print(f"\n  {'Dimension':<25s} {'Mean':>6s}  {'Min':>4s}  {'Max':>4s}")
print(f"  {'-'*25} {'-'*6}  {'-'*4}  {'-'*4}")
for d in dims:
    vals = [ex["quality_scores"][d] for ex in scored]
    print(f"  {d:<25s} {np.mean(vals):6.2f}  {min(vals):4.0f}  {max(vals):4.0f}")

# Identify which dimensions caused most removals
removed_exs = [ex for ex in scored if ex["quality_scores"]["overall"] < QUALITY_THRESHOLD]
if removed_exs:
    print(f"\n  Low-score breakdown (removed examples):")
    for d in dims:
        low = sum(1 for ex in removed_exs if ex["quality_scores"][d] <= 2)
        print(f"    {d:<25s}: {low} examples scored ≤ 2")

# Replace sft_data with filtered version
sft_data = sft_data_filtered

# Re-save filtered data
with jsonlines.open(SFT_OUTPUT_FILE, mode='w') as writer:
    writer.write_all(sft_data)
print(f"\n💾 Filtered dataset saved to {SFT_OUTPUT_FILE}")

# ── Visualisation: histogram of quality scores ───────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

all_dims = dims + ["overall"]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336", "#607D8B"]

for idx, d in enumerate(all_dims):
    ax = axes[idx]
    vals = [ex["quality_scores"][d] for ex in scored]
    ax.hist(vals, bins=np.arange(0.5, 6.5, 1), color=colors[idx], edgecolor="white", alpha=0.85)
    ax.set_title(d.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel("Score")
    ax.set_ylabel("Count")
    ax.set_xticks(range(1, 6))
    ax.axvline(QUALITY_THRESHOLD, color="red", linestyle="--", linewidth=1, label=f"Threshold ({QUALITY_THRESHOLD})")
    if d == "overall":
        ax.legend(fontsize=9)

fig.suptitle("SFT Data Quality Score Distribution (all generated examples)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "quality_scores_histogram.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"📈 Histogram saved to {OUTPUT_DIR / 'quality_scores_histogram.png'}")

## 7. DPO Preference Data Generation

For DPO (Direct Preference Optimization), we need pairs of:
- **Chosen response:** The pedagogically correct tutor response (scaffolding, co-regulation, short sentences)
- **Rejected response:** A response that violates our principles (giving answers, ignoring emotions, wall of text)

We generate these by asking the teacher model to produce BOTH a good and bad response to the same student message.

In [25]:
def build_dpo_generation_prompt(
    profile: LearnerProfile,
    scenario: AcademicScenario,
    zone: Zone,
) -> str:
    """Build a prompt that generates chosen/rejected response pairs."""
    student_cue = random.choice(ZONE_STUDENT_CUES[zone])

    return f"""You are an expert in neurodiversity-affirming education. Generate a DPO training pair for an AI tutor.

**Context:**
- Student: {profile.age_range} year-old with {profile.condition}
- Subject: {scenario.subject} — {scenario.topic} ({scenario.grade_level})
- Problem: {scenario.problem}
- Student's current emotional state: {zone.value.upper()} zone
- Student just said: "{student_cue}"

Generate TWO responses to this student message:

**CHOSEN (ideal response):** A neurodiversity-affirming response that:
- Addresses the emotional state FIRST if not in Green zone
- Uses short sentences (under 15 words each)
- Uses bullet points or numbered steps for clarity
- Does NOT give the answer — scaffolds toward discovery
- Validates the student's effort or feelings
- Checks in: "How does that feel?" or "Does that make sense?"

**REJECTED (bad response):** A response that violates pedagogical best practices:
- Ignores the student's emotional state
- Uses long, dense paragraphs
- Gives the answer directly without scaffolding
- Says things like "This is easy" or "Just try harder"
- Uses complex vocabulary inappropriate for the age group
- Moves forward despite signs of distress

**Output Format (EXACTLY):**

STUDENT_MESSAGE: [the student's message]
CHOSEN: [the ideal tutor response]
REJECTED: [the bad tutor response]

Generate now."""


def parse_dpo_pair(raw_text: str, student_msg_override: str = None) -> dict | None:
    """Parse a DPO pair from teacher model output."""
    try:
        # Extract sections
        student_match = re.search(r'STUDENT_MESSAGE:\s*(.+?)(?=CHOSEN:)', raw_text, re.DOTALL)
        chosen_match = re.search(r'CHOSEN:\s*(.+?)(?=REJECTED:)', raw_text, re.DOTALL)
        rejected_match = re.search(r'REJECTED:\s*(.+?)$', raw_text, re.DOTALL)

        if not all([student_match, chosen_match, rejected_match]):
            return None

        student_msg = student_match.group(1).strip()
        chosen = chosen_match.group(1).strip()
        rejected = rejected_match.group(1).strip()

        # Basic quality checks
        if len(chosen) < 20 or len(rejected) < 20:
            return None

        return {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": student_msg},
            ],
            "chosen": [{"role": "assistant", "content": chosen}],
            "rejected": [{"role": "assistant", "content": rejected}],
        }
    except Exception:
        return None


print("DPO generation and parsing functions defined.")

DPO generation and parsing functions defined.


In [30]:
DPO_OUTPUT_FILE = OUTPUT_DIR / "dpo_dataset.jsonl"
DPO_VARIATIONS = 1  # pairs per combo


def generate_dpo_example(
    profile: LearnerProfile,
    scenario: AcademicScenario,
    zone: Zone,
) -> dict | None:
    """Generate a single DPO preference pair using the LOCAL model."""
    prompt_text = build_dpo_generation_prompt(profile, scenario, zone)
    # Pass the multimodal content list directly to generate_text
    multimodal_prompt = [{"type": "text", "text": prompt_text}]

    try:
        raw = generate_text(
            multimodal_prompt,
            max_new_tokens=1500,
            temperature=0.9,  # Higher temp for DPO = more diversity in rejected responses
            top_p=0.95,
        )

        pair = parse_dpo_pair(raw)

        if pair is not None:
            pair["metadata"] = {
                "profile": profile.condition,
                "subject": scenario.subject,
                "topic": scenario.topic,
                "zone": zone.value,
            }
        return pair

    except Exception as e:
        print(f"  [ERROR] {e}")
        return None


def generate_dpo_dataset():
    """Generate the full DPO preference dataset (sequential, local GPU)."""
    combos = []
    for profile in PROFILES:
        for scenario in SCENARIOS:
            for zone in Zone:
                for _ in range(DPO_VARIATIONS):
                    combos.append((profile, scenario, zone))

    random.shuffle(combos)
    print(f"Total DPO generation tasks: {len(combos)}")

    results = []
    failed = 0

    for profile, scenario, zone in tqdm(combos, desc="Generating DPO pairs"):
        result = generate_dpo_example(profile, scenario, zone)
        if result is not None:
            results.append(result)
        else:
            failed += 1

        # Save checkpoint every 50 examples
        if len(results) % 50 == 0 and len(results) > 0:
            with jsonlines.open(DPO_OUTPUT_FILE, mode='w') as writer:
                writer.write_all(results)

    # Final save
    with jsonlines.open(DPO_OUTPUT_FILE, mode='w') as writer:
        writer.write_all(results)

    print(f"\n✅ Generated {len(results)} DPO pairs ({failed} failed)")
    print(f"   Saved to: {DPO_OUTPUT_FILE}")

    # Stats
    if results:
        avg_chosen_len = sum(len(r['chosen'][0]['content']) for r in results) / len(results)
        avg_rejected_len = sum(len(r['rejected'][0]['content']) for r in results) / len(results)
        print(f"   Avg chosen length: {avg_chosen_len:.0f} chars")
        print(f"   Avg rejected length: {avg_rejected_len:.0f} chars")

    return results


dpo_data = generate_dpo_dataset()


Total DPO generation tasks: 384


Generating DPO pairs:   0%|          | 0/384 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


KeyboardInterrupt: 

In [27]:
# Debugging cell to inspect raw model output for DPO
test_profile = PROFILES[0]
test_scenario = SCENARIOS[0]
test_zone = Zone.YELLOW

prompt_text = build_dpo_generation_prompt(test_profile, test_scenario, test_zone)
multimodal_prompt = [{"type": "text", "text": prompt_text}]

print("Generating raw text... this may take a moment.")
try:
    raw_output = generate_text(
        multimodal_prompt,
        max_new_tokens=1500,
        temperature=0.9,
        top_p=0.95,
    )
    print("\n" + "="*60)
    print("RAW OUTPUT:")
    print("="*60)
    print(raw_output)
    print("="*60)

    parsed = parse_dpo_pair(raw_output)
    if parsed:
        print("\n✅ Parser succeeded!")
    else:
        print("\n❌ Parser failed. Check if the raw output exactly matches the expected markers: STUDENT_MESSAGE:, CHOSEN:, and REJECTED:")
except Exception as e:
    import traceback
    print(f"\nError during generation:")
    traceback.print_exc()

Generating raw text... this may take a moment.

Error during generation:


Traceback (most recent call last):
  File "/tmp/ipykernel_2922/1699768847.py", line 11, in <cell line: 0>
    raw_output = generate_text(
                 ^^^^^^^^^^^^^^
  File "/content/castalia/hackathons/gemma-4-good-hackathon/EduTutor/local_model.py", line 234, in generate_text
    inputs = _tokenizer.apply_chat_template(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/processing_utils.py", line 1807, in apply_chat_template
    visuals = [content for content in message["content"] if content["type"] in ["image", "video"]]
                                                            ~~~~~~~^^^^^^^^
KeyError: 'type'


In [28]:
import inspect
from local_model import generate_text

print(inspect.getsource(generate_text))

def generate_text(
    prompt: str,
    max_new_tokens: int = 2048,
    temperature: float = 0.9,
    top_p: float = 0.95,
    system_prompt: str = None,
) -> str:
    """Generate text from the locally loaded model.
    
    This is the drop-in replacement for:
        client.aio.models.generate_content(model=MODEL, contents=prompt)
    
    Args:
        prompt: The user prompt / instruction.
        max_new_tokens: Max output length.
        temperature: Sampling temperature (higher = more diverse).
        top_p: Nucleus sampling threshold.
        system_prompt: Optional system prompt to prepend.
    
    Returns:
        Generated text string.
    """
    assert _model is not None, "Call load_teacher_model() first!"
    
    if system_prompt:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ]
    else:
        messages = [{"role": "user", "content": prompt}]
    
    inputs = _tokenizer.apply_ch

## 8. Data Quality Checks

Before handing off to fine-tuning, we validate that our generated data meets quality standards.

In [ ]:
import statistics


def quality_report(sft_path: Path, dpo_path: Path):
    """Generate a quality report on the generated datasets."""
    print("=" * 60)
    print("📊 DATASET QUALITY REPORT")
    print("=" * 60)

    # --- SFT Dataset ---
    sft_examples = []
    with jsonlines.open(sft_path) as reader:
        sft_examples = list(reader)

    print(f"\n📝 SFT Dataset: {len(sft_examples)} examples")

    turn_counts = [r['metadata']['num_turns'] for r in sft_examples]
    print(f"   Turns per conversation: min={min(turn_counts)}, max={max(turn_counts)}, "
          f"mean={statistics.mean(turn_counts):.1f}, median={statistics.median(turn_counts):.1f}")

    # Check for answer-giving violations
    answer_violation_phrases = [
        "the answer is", "the correct answer", "it equals",
        "this is easy", "you should know", "just focus",
        "try harder", "calm down"
    ]
    violations = 0
    for ex in sft_examples:
        for msg in ex['messages']:
            if msg['role'] == 'assistant':
                content_lower = msg['content'].lower()
                if any(phrase in content_lower for phrase in answer_violation_phrases):
                    violations += 1
                    break

    violation_pct = (violations / len(sft_examples)) * 100 if sft_examples else 0
    print(f"   ⚠️  Pedagogical violations detected: {violations} ({violation_pct:.1f}%)")
    if violation_pct > 5:
        print("   🔴 WARNING: Violation rate too high — consider filtering these examples")
    else:
        print("   🟢 Violation rate acceptable")

    # Check sentence length in assistant responses
    sentence_lengths = []
    for ex in sft_examples:
        for msg in ex['messages']:
            if msg['role'] == 'assistant':
                sentences = re.split(r'[.!?]+', msg['content'])
                for s in sentences:
                    words = s.strip().split()
                    if len(words) > 2:  # skip trivial fragments
                        sentence_lengths.append(len(words))

    if sentence_lengths:
        avg_sent_len = statistics.mean(sentence_lengths)
        print(f"   📏 Avg sentence length: {avg_sent_len:.1f} words "
              f"(target: <15, {'🟢 GOOD' if avg_sent_len < 15 else '🟡 REVIEW'})")

    # --- DPO Dataset ---
    dpo_examples = []
    with jsonlines.open(dpo_path) as reader:
        dpo_examples = list(reader)

    print(f"\n🔀 DPO Dataset: {len(dpo_examples)} preference pairs")

    # Ensure chosen is meaningfully different from rejected
    same_count = 0
    for ex in dpo_examples:
        if ex['chosen'][0]['content'].strip() == ex['rejected'][0]['content'].strip():
            same_count += 1

    print(f"   Identical chosen/rejected: {same_count} (should be 0)")

    # Length comparison — chosen should generally be shorter (concise, chunked)
    chosen_lens = [len(ex['chosen'][0]['content']) for ex in dpo_examples]
    rejected_lens = [len(ex['rejected'][0]['content']) for ex in dpo_examples]
    print(f"   Avg chosen length: {statistics.mean(chosen_lens):.0f} chars")
    print(f"   Avg rejected length: {statistics.mean(rejected_lens):.0f} chars")

    print("\n" + "=" * 60)
    print(f"✅ Total training examples: {len(sft_examples)} SFT + {len(dpo_examples)} DPO")
    print("=" * 60)


quality_report(SFT_OUTPUT_FILE, DPO_OUTPUT_FILE)

### 8.1 Data Distribution Analysis

Visualise the **balance** of our dataset across the two key axes — **learner profile** and **emotional zone**.
A well-balanced dataset ensures the fine-tuned model doesn't overfit to any single profile or zone.

- **Heatmap:** Average conversation length (turns) per profile × zone cell
- **Bar chart:** Total example counts per profile and per zone, side by side

In [ ]:
# ── Profile x Zone heatmap ──────────────────────────────────────
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

PROFILE_SHORT = {
    "ADHD (combined type)": "ADHD",
    "Autism Spectrum (Level 1)": "Autism",
    "Dyslexia": "Dyslexia",
    "Dyscalculia": "Dyscalculia",
}
ZONE_ORDER = ["green", "yellow", "red", "blue"]
PROFILE_ORDER = ["ADHD", "Autism", "Dyslexia", "Dyscalculia"]

# Build a dict  (short_profile, zone) -> list of num_turns
from collections import defaultdict
cell_turns = defaultdict(list)
for ex in sft_data:
    m = ex["metadata"]
    short = PROFILE_SHORT.get(m["profile"], m["profile"])
    cell_turns[(short, m["zone"])].append(m["num_turns"])

# Build matrix  (rows = profiles, cols = zones)
heat_matrix = np.zeros((len(PROFILE_ORDER), len(ZONE_ORDER)))
for ri, prof in enumerate(PROFILE_ORDER):
    for ci, zone in enumerate(ZONE_ORDER):
        vals = cell_turns.get((prof, zone), [])
        heat_matrix[ri, ci] = np.mean(vals) if vals else 0

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    heat_matrix,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    xticklabels=ZONE_ORDER,
    yticklabels=PROFILE_ORDER,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Dataset Balance: Conversation Length by Profile \u00d7 Zone",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Emotional Zone", fontsize=11)
ax.set_ylabel("Learner Profile", fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "data_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Heatmap saved to", OUTPUT_DIR / "data_distribution.png")

# ── Bar chart: total examples per profile & per zone ────────────
profile_counts = defaultdict(int)
zone_counts = defaultdict(int)
for ex in sft_data:
    m = ex["metadata"]
    short = PROFILE_SHORT.get(m["profile"], m["profile"])
    profile_counts[short] += 1
    zone_counts[m["zone"]] += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Per profile
prof_vals = [profile_counts.get(p, 0) for p in PROFILE_ORDER]
ax1.bar(PROFILE_ORDER, prof_vals, color=["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"],
        edgecolor="white")
ax1.set_title("Examples per Learner Profile", fontsize=12, fontweight="bold")
ax1.set_ylabel("Count")
for i, v in enumerate(prof_vals):
    ax1.text(i, v + 0.5, str(v), ha="center", fontweight="bold")

# Per zone
zone_vals = [zone_counts.get(z, 0) for z in ZONE_ORDER]
zone_colors = ["#4CAF50", "#FFC107", "#F44336", "#2196F3"]
ax2.bar(ZONE_ORDER, zone_vals, color=zone_colors, edgecolor="white")
ax2.set_title("Examples per Emotional Zone", fontsize=12, fontweight="bold")
ax2.set_ylabel("Count")
for i, v in enumerate(zone_vals):
    ax2.text(i, v + 0.5, str(v), ha="center", fontweight="bold")

fig.suptitle("Dataset Distribution Overview", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "distribution_barchart.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Bar chart saved to", OUTPUT_DIR / "distribution_barchart.png")

## 9. Export for Hugging Face Datasets

Convert our JSONL files into train/validation splits for seamless integration with the Unsloth training pipeline in Notebook 2.

In [ ]:
from datasets import Dataset


def load_and_split(path: Path, test_size: float = 0.1) -> dict:
    """Load JSONL and create train/validation split."""
    with jsonlines.open(path) as reader:
        data = list(reader)

    random.shuffle(data)
    split_idx = int(len(data) * (1 - test_size))

    return {
        "train": data[:split_idx],
        "validation": data[split_idx:],
    }


# SFT split
sft_splits = load_and_split(SFT_OUTPUT_FILE)
print(f"SFT: {len(sft_splits['train'])} train / {len(sft_splits['validation'])} validation")

# DPO split
dpo_splits = load_and_split(DPO_OUTPUT_FILE)
print(f"DPO: {len(dpo_splits['train'])} train / {len(dpo_splits['validation'])} validation")

# Save splits
for split_name, split_data in sft_splits.items():
    out = OUTPUT_DIR / f"sft_{split_name}.jsonl"
    with jsonlines.open(out, mode='w') as writer:
        writer.write_all(split_data)
    print(f"  Saved {out}")

for split_name, split_data in dpo_splits.items():
    out = OUTPUT_DIR / f"dpo_{split_name}.jsonl"
    with jsonlines.open(out, mode='w') as writer:
        writer.write_all(split_data)
    print(f"  Saved {out}")

## 10. Sample Inspection

Let's visually inspect a few generated examples to ensure quality before moving to fine-tuning.

In [ ]:
def display_sft_example(example: dict):
    """Pretty-print an SFT conversation."""
    meta = example.get('metadata', {})
    print(f"\n{'━' * 60}")
    print(f"📋 Profile: {meta.get('profile', 'N/A')} | "
          f"Subject: {meta.get('subject', 'N/A')} — {meta.get('topic', 'N/A')} | "
          f"Zone: {meta.get('zone', 'N/A').upper()}")
    print(f"{'━' * 60}")

    for msg in example['messages']:
        if msg['role'] == 'system':
            continue  # skip system prompt display

        icon = '🧒' if msg['role'] == 'user' else '🤖'
        label = 'Student' if msg['role'] == 'user' else 'EduTutor'
        print(f"\n{icon} **{label}:**")
        print(f"   {msg['content'][:300]}")
        if len(msg['content']) > 300:
            print(f"   ... ({len(msg['content'])} chars total)")


def display_dpo_example(example: dict):
    """Pretty-print a DPO preference pair."""
    meta = example.get('metadata', {})
    print(f"\n{'━' * 60}")
    print(f"📋 Profile: {meta.get('profile', 'N/A')} | "
          f"Zone: {meta.get('zone', 'N/A').upper()}")
    print(f"{'━' * 60}")

    student_msg = example['prompt'][-1]['content']
    print(f"\n🧒 **Student:** {student_msg}")
    print(f"\n✅ **CHOSEN (good):**")
    print(f"   {example['chosen'][0]['content'][:300]}")
    print(f"\n❌ **REJECTED (bad):**")
    print(f"   {example['rejected'][0]['content'][:300]}")


# Show 3 SFT examples
print("\n" + "=" * 60)
print("SAMPLE SFT CONVERSATIONS")
print("=" * 60)

with jsonlines.open(SFT_OUTPUT_FILE) as reader:
    for i, example in enumerate(reader):
        if i >= 3:
            break
        display_sft_example(example)

# Show 2 DPO examples
print("\n\n" + "=" * 60)
print("SAMPLE DPO PREFERENCE PAIRS")
print("=" * 60)

with jsonlines.open(DPO_OUTPUT_FILE) as reader:
    for i, example in enumerate(reader):
        if i >= 2:
            break
        display_dpo_example(example)

## Summary

This notebook has generated:

| Dataset | Approximate Size | Purpose |
|---|---|---|
| `data/sft_train.jsonl` | ~500 conversations | SFT: Teach the model the pedagogical persona and communication style |
| `data/sft_validation.jsonl` | ~50 conversations | SFT: Validation during training |
| `data/dpo_train.jsonl` | ~350 preference pairs | DPO: Align away from answer-giving and toward productive struggle |
| `data/dpo_validation.jsonl` | ~35 preference pairs | DPO: Validation during alignment |

### Coverage Matrix

| | Math | Reading | Writing | Science |
|---|---|---|---|---|
| **ADHD** | ✅ | ✅ | ✅ | ✅ |
| **Autism** | ✅ | ✅ | ✅ | ✅ |
| **Dyslexia** | ✅ | ✅ | ✅ | ✅ |
| **Dyscalculia** | ✅ | ✅ | ✅ | ✅ |

All four Zones of Regulation (Green, Yellow, Red, Blue) are represented across every combination.

### Next Step

→ **Notebook 2: `02_unsloth_tuning.ipynb`** — Fine-tune Gemma 4 E4B using Unsloth QLoRA on this dataset.

## 🧹 NeMo Curator: Automatic Quality Filtering

We integrate **NeMo Curator** here to automatically filter our newly generated SFT and DPO datasets. This ensures all examples meet our strict pedagogical guidelines (e.g. length limits to enforce Cognitive Load Management).

In [ ]:
!pip install -q nemo_curator

from nemo_curator import CuratorPipeline
from nemo_curator.filters import LengthFilter, LanguageFilter
from nemo_curator.datasets import JSONLDataset

def curate_dataset(input_path, output_path):
    import os
    if not os.path.exists(input_path):
        print(f'\u26a0\ufe0f Skipping {input_path}, file not found.')
        return
    print(f'\ud83d\udd0d Curating {input_path}...')
    dataset = JSONLDataset(input_path)
    pipeline = CuratorPipeline([
        LengthFilter(min_length=50, max_length=1500, column='response'),
        LanguageFilter(target_language='en', column='response')
    ])
    curated_dataset = pipeline.apply(dataset)
    curated_dataset.to_jsonl(output_path)
    print(f'\u2705 Saved curated data to {output_path}')

# Apply Curator to SFT and DPO datasets
curate_dataset('data/sft_train.jsonl', 'data/sft_train_nemo_curated.jsonl')
curate_dataset('data/sft_validation.jsonl', 'data/sft_validation_nemo_curated.jsonl')
curate_dataset('data/dpo_train.jsonl', 'data/dpo_train_nemo_curated.jsonl')
curate_dataset('data/dpo_validation.jsonl', 'data/dpo_validation_nemo_curated.jsonl')


## 🛡️ Real-Time Pedagogical Guardrails (NeMo Guardrails)

We use **Colang** to define rules that the AI must follow. For EduTutor, the most critical rule is: **Never give the final answer.** This file is saved and used by the agentic tutor in Notebook 04.

In [ ]:
COLANG_CONFIG = """
define user ask for answer
  "what is the answer?"
  "tell me the result"
  "just give me the number"

define flow
  user ask for answer
  tutor explain productive struggle
  tutor provide hint

define tutor explain productive struggle
  "I can't give you the answer directly, but I'm here to help you find it! Thinking it through helps your brain grow stronger. 💪"
"""

with open("pedagogical_rails.co", "w") as f:
    f.write(COLANG_CONFIG)

print("✅ Guardrails config saved to pedagogical_rails.co")


## 🧠 Synthetic Data Generation (SDG) for Math Scenarios

The **Adaptive Math Learning Playground** requires structured JSON scenarios. We use a NeMo-inspired SDG approach to generate these for the React/tldraw frontend.

In [ ]:
SCENARIO_GEN_PROMPT = """
Generate a high-fidelity JSON scenario for the 'Adaptive Math Playground' following this schema:
{
  "id": "unique_id",
  "title": "string",
  "concept": "string",
  "student_profile": "ADHD | Autism | Dyslexia | Dyscalculia",
  "emotional_zone": "Green | Yellow | Red | Blue",
  "misconception": { "type": "string", "explanation": "string" },
  "initial_prompt": "string (neurodiversity-affirming tutor dialogue)",
  "playground_state": {
    "tool": "fraction_bars | number_line | counters",
    "initial_config": {  }
  },
  "scaffolding_hints": ["string (C)", "string (R)", "string (A)"]
}

Target Topic: Equivalent Fractions. Student Profile: ADHD.
"""

# Load teacher model (Gemma 4)
model, tokenizer = load_teacher_model()

# Generate a sample scenario
scenario_json = generate_json(SCENARIO_GEN_PROMPT)
print(json.dumps(scenario_json, indent=2))


## 📊 Evaluation and Scaling

We have integrated NeMo's pipeline elements where they provide the most value for this task:
- **NeMo Curator** filters the SFT data above.
- **NeMo Guardrails** is exported as a `.co` file for the agentic tutor UI.
- **NeMo Evaluator** can be used as a benchmarking harness to check misconception accuracy (we demonstrate an LLM-as-Judge version in Notebook 03).
- **NeMo Customizer** provides the professional scaling path for multi-node RLHF/DPO, complementing our fast single-GPU Unsloth loop.